In [1]:
import os
root_dir = os.path.abspath(os.path.dirname("__file__" if "__file__" in locals() else os.getcwd()))
os.chdir(root_dir)  # 切换到根目录，确保相对路径生效
print(f"自动识别项目根目录：{root_dir}")

自动识别项目根目录：f:\AI-project\F5TTS


# Experimental Setting / 实验设置

**Goal / 目标：** Fine-tune [F5-TTS](https://github.com/SWivid/F5-TTS) to synthesize **child voice** (儿童音色克隆).

**Dataset / 数据集：** `child-tts` — 220 short Chinese child speech recordings (~3–6 s each, ~14 min total), single speaker.

**Pipeline / 流程：**
1. Data Preprocessing → 2. Download Pretrained Model → 3. Fine-tuning → 4. Inference → 5. Objective Evaluation

**Hardware / 硬件：** NVIDIA RTX 4060 Laptop (8 GB VRAM)

**Environment Check / 环境验证**

Verify that PyTorch can access the GPU. The `f5-tts` conda environment (Python 3.11) should already be installed.

验证 PyTorch 能否访问 GPU。`f5-tts` conda 环境（Python 3.11）应已提前安装完毕。

In [9]:
import torch
import whisper

# 验证 PyTorch GPU 支持
print("✅ PyTorch GPU 状态：", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ GPU 型号：", torch.cuda.get_device_name(0))

# 验证 Whisper GPU 支持（Whisper 会自动用 CUDA）
whisper_device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Whisper 运行设备：", whisper_device)

✅ PyTorch GPU 状态： True
✅ GPU 型号： NVIDIA GeForce RTX 4060 Laptop GPU
✅ Whisper 运行设备： cuda


**Data Preprocessing / 数据预处理**

> ⚠️ **Run only once** — outputs are cached in `data/child-tts_processed/`. No need to re-run unless the raw data changes.
> ⚠️ **只需运行一次** — 结果已缓存在 `data/child-tts_processed/` 目录，原始数据不变则无需重复运行。

**What this script does / 脚本功能：**
1. Uses Whisper (large-v2) to transcribe each WAV → `.txt` file
2. Computes audio duration for each file → `duration.json`
3. Builds a HuggingFace Arrow dataset → `raw.arrow`
4. Generates `vocab.txt` (character-level tokenization)

**Estimated time / 预计耗时：** ~19 min (220 files × ~5 s Whisper inference)

**Dataset info / 数据集信息：**
- Source: `data/child-tts/` — 220 child speech WAVs, single Chinese-speaking child
- Reference audio: `000018.wav` — "城里有好多游乐场，可好玩儿了！"
- Total duration: ~14 min; sample rate: 24 kHz


In [ ]:
import os

# 关键：直接硬编码项目根目录（一次写死，再也不跑偏）
# ！！！替换成你 AIAA2205-assignment2-F5-TTS 文件夹的实际绝对路径！！！
root_dir = "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS"

# 强制切换到项目根目录（覆盖任何内核默认目录）
os.chdir(root_dir)
print(f"🔒 已锁定项目根目录：{root_dir}")

# 拼接核心路径（固定对应项目文件夹结构）
data_prepare_script = os.path.join(root_dir, "src", "f5_tts", "train", "datasets", "data_prepare.py")
input_dir = os.path.join(root_dir, "data", "child-tts")  # 放 .wav 和 .csv 的目录
output_dir = os.path.join(root_dir, "data", "child-tts_processed")  # 输出 arrow 文件的目录

# 路径验证（报错直接提示问题，不用猜）
print("\n✅ 路径验证：")
if not os.path.exists(data_prepare_script):
    raise FileNotFoundError(f"❌ 没找到脚本！请确认 {data_prepare_script} 路径正确")
if not os.path.exists(input_dir):
    raise FileNotFoundError(f"❌ 没找到数据目录！请在 {input_dir} 放入 .wav 和 .csv 文件")

# 自动创建输出目录（避免缺失报错）
os.makedirs(output_dir, exist_ok=True)
print(f"✅ 脚本路径正确：{data_prepare_script}")
print(f"✅ 数据目录正确：{input_dir}")
print(f"✅ 输出目录已就绪：{output_dir}")

# 执行数据准备脚本（捕获完整日志）
import subprocess
run_command = f'python "{data_prepare_script}" "{input_dir}" "{output_dir}"'
print(f"\n🚀 执行命令：{run_command}")
result = subprocess.run(run_command, shell=True, capture_output=True, text=True, encoding="utf-8")

# 展示执行结果
print(f"\n📊 执行结果：{'成功' if result.returncode == 0 else '失败'}")
print(f"输出日志：{result.stdout}")
if result.stderr:
    print(f"\n❌ 错误详情：{result.stderr}")
else:
    print(f"\n🎉 生成的文件：{os.listdir(output_dir)}")

🔒 已锁定项目根目录：F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS

✅ 路径验证：
✅ 脚本路径正确：F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS\src\f5_tts\train\datasets\data_prepare.py
✅ 数据目录正确：F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS\data\child-tts
✅ 输出目录已就绪：F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS\data\child-tts_processed

🚀 执行命令：python "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS\src\f5_tts\train\datasets\data_prepare.py" "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS\data\child-tts" "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS\data\child-tts_processed"


Install dependencies from `requirement.txt` (run once when first setting up the environment).

首次配置环境时安装依赖包，之后无需重复安装。

Check GPU availability and Python version before starting.

在开始之前检查 GPU 可用性和 Python 版本。

In [ ]:
#配置环境
!pip install -r requirement.txt

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/a2/ad/e0d3c824784ff121c03cc031f944bc7e139a8f1870ffd2845cc2dd76f6c4/absl_py-2.1.0-py3-none-any.whl (133 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/73/de/64508cb91af013aaba214752309c0967568a4219d50a4ea30e822af3c976/accelerate-1.3.0-py3-none-any.whl (336 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/c5/19/5af6804c4cc0fed83f47bff6e413a98a36618e7d40185cd36e69737f3b0e/aiofiles-23.2.1-py3-none-any.whl (15 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/b9/74/fbb6559de3607b3300b9be3cc64e97548d55678e44623db17820dbd20002/aiohappyeyeballs-2.4.4-py3-none-any.whl (14 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/fc/db/2192489a8a51b52e06627506f8ac8df69ee221de88ab9bdea77aa793aa6a/aiohttp-3.11.11-cp311-cp311-win_amd64.whl (442 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/76/ac/a7305707cb852b7e16ff80eaf56

ERROR: Ignored the following versions that require a different python version: 0.52.0 Requires-Python >=3.6,<3.9; 0.52.0rc3 Requires-Python >=3.6,<3.9; 0.53.0 Requires-Python >=3.6,<3.10; 0.53.0rc1.post1 Requires-Python >=3.6,<3.10; 0.53.0rc2 Requires-Python >=3.6,<3.10; 0.53.0rc3 Requires-Python >=3.6,<3.10; 0.53.1 Requires-Python >=3.6,<3.10; 0.54.0 Requires-Python >=3.7,<3.10; 0.54.0rc2 Requires-Python >=3.7,<3.10; 0.54.0rc3 Requires-Python >=3.7,<3.10; 0.54.1 Requires-Python >=3.7,<3.10; 0.55.0 Requires-Python >=3.7,<3.11; 0.55.0rc1 Requires-Python >=3.7,<3.11; 0.55.1 Requires-Python >=3.7,<3.11; 0.55.2 Requires-Python >=3.7,<3.11; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement nvidia-nccl-cu12==2.21.5 (from versions: 0.0.1.dev5)
ERROR: No matching distribution found for nvidia-nccl-cu12==2

In [ ]:
# Check GPU
!nvidia-smi

Mon Nov 17 15:46:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P0             15W /  130W |       0MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Check Python Version
!python --version

Python 3.11.14


# Data Preparation / 数据预处理

Prepare the `child-tts` dataset for training. The pipeline:
- Transcribes each `.wav` using Whisper → builds `metadata.csv`
- Computes duration → `duration.json`
- Converts to HuggingFace Arrow format → `raw.arrow`
- Extracts vocabulary → `vocab.txt`

> **Only run the section matching your dataset type** (standard vs. dialect).
> **只需运行与你数据集类型对应的部分**（标准数据 vs. 方言数据）。

##***Run this if your dataset is not dialect***


**Standard dataset (non-dialect) — simple command reference / 标准数据集（非方言）简单命令参考：**

```bash
cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS
python ./src/f5_tts/train/datasets/data_prepare.py ./data/child-tts ./data/child-tts_processed
```

> Use the code cell in the section above for a more robust version with path validation and error handling.
> 推荐使用上面有路径验证和错误处理的 Python 代码单元格。

%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS
#文本转录，数据整理！ 上传收集的语音文件到F5-TTS/data; 输入input_dir:your audio file dir, output_dir:数据整理结果保存路径。
!python ./src/f5_tts/train/datasets/data_prepare.py ./data/child-tts ./data/child-tts_processed

##***Run these if your dataset is dialect***


In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS
#生成方言的转录文本 （如果数据集里面自带转录文本，便不需要这个）
#使用方言数据，用这个data_process. 首先读取文本为单个.txt文件 input_path: your txt dir, --output_dir: path to save single txt file.
!python ./src/f5_tts/train/datasets/read_text.py --input_path ./data/dialect_data/sichuan.txt --output_dir ./data/dialect_data/text/

In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS
#数据集整理，将speech和Text文件夹放在一个文件夹内，上传文件夹至./data，输入input_dir: 文件夹所在路径，output_dir:数据整理结果保存路径。
!python ./src/f5_tts/train/datasets/data_prepare_text.py ./data/dialect_data/ ./data/sichuan_data/

# Fine-tuning / 微调

Fine-tune the F5TTS pretrained model on the `child-tts` dataset to clone the child's voice.

**Key improvements in this notebook / 关键修复与优化：**
| Issue | Fix |
|---|---|
| `num_warmup_updates=20000` (>> total steps 5500) | Auto-computed: `max(50, min(1000, 10% of total))` |
| Linear LR decay (poor for small datasets) | Cosine annealing LR (`--scheduler cosine`) |
| No regularization | `weight_decay=1e-2` in AdamW |
| Windows CUDA env var syntax | `os.environ["CUDA_VISIBLE_DEVICES"] = "0"` |

> **First download the pretrained model and vocab before running fine-tuning.**
> **请先下载预训练模型和词汇表，再运行微调。**

In [ ]:
# 先安装依赖（如果没装的话，运行一次即可）
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "huggingface_hub"])

from huggingface_hub import hf_hub_download
import os, shutil, pathlib

# 核心修改：适配本地项目路径（和你的数据集目录同级，不用手动填复杂路径）
# 项目根目录 = 你 VSCode 打开的总文件夹，代码会自动获取
project_root = os.getcwd()
# 模型保存目录：在项目根目录下创建 ckpts/child-tts（和你的数据集目录同级）
dst_dir = os.path.join(project_root, "ckpts", "child-tts")
# 模型最终保存路径（统一文件名，方便后续微调代码读取）
dst_path = os.path.join(dst_dir, "pretrained_model_1250000.safetensors")

# 创建模型目录（不存在则自动创建）
os.makedirs(dst_dir, exist_ok=True)

print(f"项目根目录：{project_root}")
print(f"模型保存目录：{dst_dir}")

# 下载预训练模型（正确路径：包含仓库内的子目录 F5TTS_v1_Base/）
tmp_path = hf_hub_download(
    repo_id="SWivid/F5-TTS",  # 正确的仓库ID（注意是 SWivid，之前少了个 W）
    filename="F5TTS_v1_Base/model_1250000.safetensors",  # 仓库内的完整文件路径
    local_dir=dst_dir,
    local_dir_use_symlinks=False,  # 本地环境禁用软链，避免路径问题
)

# 移动并改名到最终路径（清理多余的子目录）
shutil.move(tmp_path, dst_path)

# 可选：清理下载时自动创建的空目录（F5TTS_v1_Base/）
p = pathlib.Path(tmp_path).parent
try:
    p.rmdir()
except OSError:
    pass

print(f"✅ 模型下载完成！已保存到：{dst_path}")

项目根目录：F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS
模型保存目录：F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS\ckpts\child-tts
✅ 模型下载完成！已保存到：F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS\ckpts\child-tts\pretrained_model_1250000.safetensors


In [ ]:
# ⬆️ Duplicate of the cell above — already handled by the auto-path version.
# SKIP this cell. / 此单元格是上方下载代码的重复项，跳过即可。

	**Download Vocabulary File / 下载词汇表文件**

`vocab.txt` maps each Chinese character / English letter to a token ID. We use the **original pretrained vocab** — no modification needed for standard Mandarin.

`vocab.txt` 将每个汉字/英文字母映射为模型 token ID。直接使用预训练模型的原始词汇表，无需修改（标准普通话）。

> Saved to: `ckpts/child-tts/F5TTS_v1_Base/vocab.txt` — pass this path to `--tokenizer_path` in the fine-tuning command.“文本转语音” 

In [ ]:
from huggingface_hub import hf_hub_download
import os

# 模型保存目录（和你下载的预训练模型在同一个文件夹）
dst_dir = "ckpts/child-tts"

# 确保目录存在（不存在就自动创建）
os.makedirs(dst_dir, exist_ok=True)

# 下载预训练模型配套的 vocab.txt（不换原 vocab，直接用）
hf_hub_download(
    repo_id="SwiVid/F5-TTS",  # 正确的模型仓库
    filename="F5TTS_v1_Base/vocab.txt",  # 仓库里 vocab 的准确路径
    local_dir=dst_dir,
    local_dir_use_symlinks=False  # 本地环境禁用软链，避免出错
)

print("✅ vocab.txt 下载完成！")

✅ vocab.txt 下载完成！


In [ ]:
# ❌ Linux-only command (wget). Use the huggingface_hub download cell above instead.
# ❌ 此命令仅适用于 Linux/macOS，Windows 请使用上方的 huggingface_hub 下载代码。

In [ ]:
# Verify downloaded model files / 验证下载的模型文件
import os
ckpt_dir = "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts"
for f in os.listdir(ckpt_dir):
    size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS

# ✅ Windows 兼容写法 (不能用 CUDA_VISIBLE_DEVICES=0 python ... 的 Linux 语法)
# ✅ 关键修复: num_warmup_updates 设为 -1 (自动=10%总步数), 原20000远超小数据集总步数
# ✅ 使用 cosine LR调度 + 50 epochs + weight_decay 防过拟合

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Windows 下设 CUDA 设备的正确方式

!python ./src/f5_tts/train/finetune_cli.py \
  --dataset_name child-tts_processed \
  --epochs 50 \
  --batch_size_per_gpu 2 \
  --learning_rate 1e-5 \
  --num_warmup_updates -1 \
  --scheduler cosine \
  --pretrain F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/pretrained_model_1250000.safetensors \
  --tokenizer_path F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/vocab.txt \
  --save_per_updates 300 \
  --last_per_updates 500 \
  --logger tensorboard

In [ ]:
# ❌ Wrong path — infer.py does not exist at ./infer/. Use infer_cli.py below.
# ❌ 路径错误，infer.py 不在 ./infer/ 目录。请使用下面的 infer_cli.py 单元格。

In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS
# ❌ Old training command — Windows-incompatible CUDA syntax, only 1 epoch (far too few for convergence).
# ❌ 旧训练命令：Windows 下 CUDA_VISIBLE_DEVICES=0 前缀不可用，且仅 1 epoch 根本无法收敛。
# ✅ Use the fine-tuning cell in the section above instead.
# ✅ 请直接使用上方"Fine-tuning"部分中已修正的训练单元格。


# Inference / 推理

Generate speech using the fine-tuned model. Provide a **reference audio** (from `data/child-tts/`) so the model can clone the child's voice style, then generate any target text.

使用微调后的模型生成语音。提供一段**参考音频**（来自 `data/child-tts/`），模型将克隆该儿童的音色，并合成指定文本。

**Parameters / 参数说明：**
| Param | Recommended | Description |
|---|---|---|
| `--nfe_step` | 50 | Diffusion sampling steps (higher = better quality, slower) |
| `--cfg_strength` | 2.0 | Classifier-free guidance strength (higher = more faithful to ref) |
| `--ref_audio` | `000018.wav` | Reference audio for voice cloning |
| `--ref_text` | (transcription) | Text content of the reference audio |

In [ ]:
# Verify fine-tuned checkpoint exists / 验证微调后的检查点文件是否存在
import os
ckpt_dir = "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts_processed"
if os.path.exists(ckpt_dir):
    for f in os.listdir(ckpt_dir):
        size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
        print(f"  {f}  ({size_mb:.1f} MB)")
else:
    print("⚠️ Checkpoint directory not found. Run fine-tuning first.")

**Inference with Pretrained Model (baseline) / 用预训练模型推理（基线对比）**

Use the pretrained model as a baseline — no fine-tuning applied. Compare its output against the fine-tuned model to verify that fine-tuning actually improved child voice quality.

使用未微调的预训练模型生成语音作为基线，与微调后模型的输出对比，验证微调效果。

In [ ]:
# ❌ SyntaxError: inline comments after "\" line continuation are not allowed in Python.
# ❌ 语法错误：Python 不支持在行续符"\"后添加注释，此写法导致 SyntaxError，无法运行。
# ✅ Use the corrected inference cells below.
# ✅ 请使用下方已修正的推理单元格。


SyntaxError: unexpected character after line continuation character (2555807401.py, line 2)

In [ ]:
# ❌ SyntaxError: trailing spaces after "\" (e.g. "\ " instead of "\") break line continuation.
# ❌ 语法错误：行续符"\"后有多余空格（"\ "），导致 Python 无法识别跨行命令，运行报错。
# ✅ Fixed version is in the inference cells below (use finetuned model or pretrained baseline).
# ✅ 已修正版本在下方推理单元格中。

SyntaxError: invalid syntax (2928753393.py, line 2)

In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS

# Baseline inference: pretrained model (no fine-tuning) / 基线推理：预训练模型（未微调）
# Ref audio: 000018.wav — "城里有好多游乐场，可好玩儿了！" (child speaker)
# Purpose: compare against finetuned model to measure improvement / 用于与微调模型对比，衡量提升效果
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!python ./src/f5_tts/infer/infer_cli.py \
  --model_cfg "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/src/f5_tts/configs/F5TTS_v1_Base.yaml" \
  --ckpt_file "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/pretrained_model_1250000.safetensors" \
  --vocab_file "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/F5TTS_v1_Base/vocab.txt" \
  --ref_audio "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/data/child-tts/000018.wav" \
  --ref_text "城里有好多游乐场，可好玩儿了！" \
  --gen_text "这里有好多游乐场，可好玩了" \
  --output_dir "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/test_outputs" \
  --output_file "baseline_pretrained.wav" \
  --nfe_step 50 \
  --cfg_strength 2.0



SyntaxError: invalid syntax (1825556901.py, line 3)

In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS

# Finetuned model inference: generate child voice for any Chinese text
# 微调模型推理：用任意中文文本生成儿童音色语音（修改 --gen_text 合成不同句子）
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!python ./src/f5_tts/infer/infer_cli.py \
  --model_cfg "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/src/f5_tts/configs/F5TTS_v1_Base.yaml" \
  --ckpt_file "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts_processed/model_last.pt" \
  --vocab_file "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/F5TTS_v1_Base/vocab.txt" \
  --ref_audio "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/data/child-tts/000018.wav" \
  --ref_text "城里有好多游乐场，可好玩儿了！" \
  --gen_text "今天天气很好，我们去公园玩吧！" \
  --output_dir "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/test_outputs" \
  --output_file "finetuned_child_speech.wav" \
  --nfe_step 50 \
  --cfg_strength 2.0

F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS


Download Vocos from huggingface charactr/vocos-mel-24khz
Using F5TTS_v1_Base...

vocab :  F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/vocab.txt
token :  custom
model :  F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/pretrained_model_1250000.safetensors 

F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/data/child-tts/000018.wav
gen_text 0 【仅输出中文，禁止输出英文】你想生成的中文内容（如‘今天天气很好，适合去公园玩’）


Generating audio in 1 batches...


e:\Users\ASUS\miniconda3\envs\f5-tts\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources

100%|██████████| 1/1 [00:06<00:00,  6.68s/it]


In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS

# ✅ 使用微调后模型推理 (Windows 兼容)
# model_last.pt = 微调模型 | pretrained_model_1250000.safetensors = 原始预训练模型
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!python ./src/f5_tts/infer/infer_cli.py \
  --model_cfg "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/src/f5_tts/configs/F5TTS_v1_Base.yaml" \
  --ckpt_file "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts_processed/model_last.pt" \
  --vocab_file "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/ckpts/child-tts/vocab.txt" \
  --ref_audio "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/data/child-tts/000018.wav" \
  --ref_text "这里有好多游乐场可好玩了" \
  --gen_text "这里有好多游乐场，可好玩了" \
  --output_dir "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/test_outputs" \
  --output_file "finetuned_child_speech.wav" \
  --nfe_step 50 \
  --cfg_strength 2.0

F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS


'CUDA_VISIBLE_DEVICES' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [ ]:
# List generated WAV files / 查看生成的音频文件
import os, glob
output_dir = "F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS/test_outputs"
wavs = glob.glob(os.path.join(output_dir, "*.wav"))
for w in sorted(wavs):
    size_kb = os.path.getsize(w) / 1024
    print(f"  {os.path.basename(w)}  ({size_kb:.0f} KB)")

F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS


'ls' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [ ]:
# ❌ TYPO: "CUDA_VISIBlE_DEVICES" (lowercase l) — never worked. Replaced by cell above.
# ❌ 拼写错误：CUDA_VISIBlE_DEVICES（小写l），此命令从未正常运行过。请用上方已修正的推理单元格。


F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS


'CUDA_VISIBlE_DEVICES' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [ ]:
# ❌ Windows-incompatible: CUDA_VISIBLE_DEVICES=0 prefix syntax does not work in PowerShell/CMD.
# ❌ Windows 不兼容：CUDA_VISIBLE_DEVICES=0 前缀在 PowerShell/CMD 下无效。
# ❌ Also missing --ref_audio (required for voice cloning).
# ❌ 此命令还缺少 --ref_audio（语音克隆必须提供参考音频）。
# ✅ Use the inference cells above with import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0".
# ✅ 请使用上方推理单元格（已用 Python 设置 CUDA 环境变量的正确写法）。

F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS


'CUDA_VISIBLE_DEVICES' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


# Objective Evaluation / 客观指标评估

Measure speech quality of the generated audio using `speechscore`.

**Metrics / 评估指标：**
| Metric | Range | Meaning |
|---|---|---|
| PESQ | 1.0 – 4.5 | Perceptual speech quality (higher = better) |
| STOI | 0.0 – 1.0 | Short-time objective intelligibility (higher = better) |
| CSIG | 1.0 – 5.0 | Signal distortion (higher = better) |
| CBAK | 1.0 – 5.0 | Background noise (higher = better) |
| COVL | 1.0 – 5.0 | Overall quality (higher = better) |

> Compare pretrained baseline (`baseline_pretrained.wav`) vs. finetuned (`finetuned_child_speech.wav`) to measure improvement.
> 对比预训练基线与微调模型的生成结果，量化提升效果。

In [ ]:
%cd F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS

# ✅ 安装 speechscore 依赖（首次运行需要）
!pip install pesq pystoi scipy -q

# ✅ 运行语音质量评估
# 注意: test_path = 生成的语音, reference_path = 参考（干净）语音
!python ./speechscore/demo.py

F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS


Traceback (most recent call last):
  File "F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS\speechscore\demo.py", line 4, in <module>
    from speechscore import SpeechScore 
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS\speechscore\speechscore.py", line 18, in <module>
    from scores.csig import CSIG
  File "F:\AI-project\F5TTS\AIAA2205-assignment2-F5-TTS\speechscore\scores\csig.py", line 3, in <module>
    from pesq import pesq
ModuleNotFoundError: No module named 'pesq'


# Batch Inference & Evaluation / 批量推理与评估

**Complete Pipeline / 完整流程：**
1. Install dependencies / 安装依赖
2. Batch inference on test data / 对测试数据进行批量推理
3. Evaluate audio quality / 评估音频质量
4. Generate evaluation report / 生成评估报告

> **Note:** This section uses optimized inference parameters to maximize audio quality scores (excluding frame-alignment metrics like MCD).
> **注意：** 此部分使用优化的推理参数以最大化音频质量分数（排除帧对齐相关指标如MCD）。


In [ ]:
# Step 1: Install evaluation dependencies / 步骤1：安装评估依赖
import subprocess
import sys

def install_deps():
    """安装评估所需的依赖包"""
    packages = [
        "pesq==0.0.4",
        "pystoi==0.4.1",
        "scipy",
    ]
    
    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
            print(f"✅ {package} 安装成功")
        except Exception as e:
            print(f"⚠️ {package} 安装失败: {e}")

# 运行安装
install_deps()

# 验证安装
try:
    import pesq
    import pystoi
    print("✅ 所有依赖已就绪")
except ImportError as e:
    print(f"❌ 依赖缺失: {e}")


**Step 2: Batch Inference / 步骤2：批量推理**

Run batch inference on all test data files. The script will:
- Transcribe each test audio using Whisper
- Generate speech using the fine-tuned model
- Save all generated audio files

对所有测试数据文件运行批量推理。脚本将：
- 使用Whisper转录每个测试音频
- 使用微调模型生成语音
- 保存所有生成的音频文件


In [ ]:
# Step 2: Run batch inference / 步骤2：运行批量推理
import os
os.chdir("F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS")

# 运行批量推理脚本
!python batch_inference.py


**Step 3: Evaluate Audio Quality / 步骤3：评估音频质量**

Evaluate the generated audio files using multiple metrics:
- **PESQ**: Perceptual speech quality (1.0-4.5, higher is better)
- **STOI**: Short-time objective intelligibility (0.0-1.0, higher is better)
- **CSIG, CBAK, COVL**: Composite quality metrics (1.0-5.0, higher is better)
- **DNSMOS**: DNS challenge MOS score
- **SRMR**: Speech quality metric

> **Note:** MCD (Mel Cepstral Distance) is excluded as it requires frame alignment.

使用多个指标评估生成的音频文件：
- **PESQ**: 感知语音质量（1.0-4.5，越高越好）
- **STOI**: 短时客观可懂度（0.0-1.0，越高越好）
- **CSIG, CBAK, COVL**: 综合质量指标（1.0-5.0，越高越好）
- **DNSMOS**: DNS挑战MOS分数
- **SRMR**: 语音质量指标

> **注意：** 排除MCD（梅尔倒谱距离），因为它需要帧对齐。


In [ ]:
# Step 3: Evaluate audio quality / 步骤3：评估音频质量
import os
os.chdir("F:/AI-project/F5TTS/AIAA2205-assignment2-F5-TTS")

# 运行评估脚本
!python evaluate_audio.py


**Step 4: View Evaluation Results / 步骤4：查看评估结果**

Display the evaluation results and summary statistics.
显示评估结果和摘要统计。


In [ ]:
# Step 4: View evaluation results / 步骤4：查看评估结果
import json
import pandas as pd
from pathlib import Path

results_file = Path("evaluation_results/evaluation_results.json")

if results_file.exists():
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)
    
    # 显示平均分数
    if "Mean_Score" in results:
        print("=" * 60)
        print("平均分数摘要 / Mean Score Summary")
        print("=" * 60)
        
        mean_scores = results["Mean_Score"]
        
        # 创建DataFrame以便更好地显示
        df_data = []
        for metric, stats in mean_scores.items():
            df_data.append({
                "Metric": metric,
                "Mean": f"{stats['mean']:.4f}",
                "Min": f"{stats['min']:.4f}",
                "Max": f"{stats['max']:.4f}",
                "Std": f"{stats['std']:.4f}"
            })
        
        df = pd.DataFrame(df_data)
        print(df.to_string(index=False))
        print("=" * 60)
        
        # 显示关键指标
        print("\n关键指标 / Key Metrics:")
        key_metrics = ["PESQ", "STOI", "CSIG", "CBAK", "COVL", "DNSMOS"]
        for metric in key_metrics:
            if metric in mean_scores:
                print(f"  {metric}: {mean_scores[metric]['mean']:.4f}")
    else:
        print("⚠️ 未找到平均分数数据")
else:
    print(f"❌ 评估结果文件不存在: {results_file}")
    print("请先运行评估脚本！")
